In [60]:
# Анализ динамики новостной повестки по RSS-лентам СМИ

**Итоговый проект программы «Аналитик данных»**

**Автор:** [Арутюнов-Григорян Армен Гарикович]
**Научный руководитель:** [Бурова Маргарита Борисовна]
**Дата:** [17.10.2026]

---

## 1. Постановка задачи

### 1.1. Проблема

Современные медиа генерируют тысячи новостей в сутки. Редакции, пресс-службы и аналитические отделы не успевают вручную отслеживать, какие темы выходят на первый план и кто задаёт тон повестки. Отсутствие системного мониторинга приводит к запаздыванию реакции на информационные волны.

### 1.2. Цель проекта

Разработать инструмент анализа динамики новостной повестки на основе RSS-лент российских СМИ, который позволяет:

- отслеживать частоту упоминания ключевых тем во времени;
- выявлять периоды всплесков и затухания интереса к сюжетам;
- сравнивать повестки разных изданий;
- формировать аналитические отчёты и визуализации.

### 1.3. Ожидаемый результат

1. Программный код на Python, реализующий полный цикл анализа.
2. Набор визуализаций: временные ряды, тепловые карты, сравнения.
3. Итоговый отчёт с выводами и рекомендациями.

### 1.4. Источники данных

RSS-ленты российских СМИ: ТАСС, РБК, Газета.ru, Lenta.ru, RT.
    

SyntaxError: invalid character '«' (U+00AB) (415962672.py, line 3)

In [61]:
## 2. Подготовка окружения

Устанавливаем зависимости и создаём структуру папок проекта.

SyntaxError: invalid syntax (3015155922.py, line 3)

In [62]:
!pip install feedparser pandas numpy matplotlib seaborn scikit-learn


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [63]:
import os

# Создание структуры папок
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("reports/figures", exist_ok=True)

print("Структура папок создана:")
print("  data/raw/")
print("  data/processed/")
print("  reports/figures/")


Структура папок создана:
  data/raw/
  data/processed/
  reports/figures/


In [ ]:
## 3. Импорт библиотек

Загружаем все необходимые библиотеки и настраиваем отображение.


In [64]:
import feedparser
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import os
import warnings
from datetime import datetime

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# Настройки отображения
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Поддержка кириллицы в графиках
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_style("whitegrid")

print("Библиотеки загружены.")

Библиотеки загружены.


In [ ]:
## 4. Сбор данных

### 4.1. Источники RSS

Определяем список RSS-лент российских СМИ. Для каждого источника указываем имя и URL.

In [65]:
def check_source(name, url):
    """Проверяет доступность RSS-источника."""
    try:
        feed = feedparser.parse(url, request_headers={"User-Agent": UA})
        return len(feed.entries), getattr(feed, "status", "n/a")
    except Exception as e:
        return 0, f"error: {e}"

working_sources = {}
print("Проверка источников:\n")
for name, url in RSS_SOURCES.items():
    count, status = check_source(name, url)
    mark = "✓" if count > 0 else "✗"
    print(f"{mark} {name:12s} HTTP {status:>5}  записей: {count}")
    if count > 0:
        working_sources[name] = url

print(f"\nРаботающих источников: {len(working_sources)}")

Проверка источников:

✗ ria          HTTP   n/a  записей: 0
✓ rt           HTTP   200  записей: 50
✓ lenta        HTTP   200  записей: 200

Работающих источников: 2


In [ ]:
### 4.3. Функции сбора и сохранения

Реализуем две функции:
- `fetch_rss` — собирает записи из одной RSS-ленты;
- `save_to_db` — сохраняет записи в базу SQLite.

In [66]:
def fetch_rss(source_name, url):
    """Собирает записи из RSS-ленты."""
    feed = feedparser.parse(url, request_headers={"User-Agent": UA})
    records = []
    for entry in feed.entries:
        records.append({
            "source": source_name,
            "title": entry.get("title", ""),
            "link": entry.get("link", ""),
            "published": entry.get("published", ""),
            "summary": entry.get("summary", ""),
            "category": entry.get("category", ""),
        })
    return records


def save_to_db(records, db_path="news.db", table="news"):
    """Сохраняет записи в SQLite."""
    if not records:
        return 0
    conn = sqlite3.connect(db_path)
    pd.DataFrame(records).to_sql(table, conn, if_exists="append", index=False)
    conn.close()
    return len(records)

print("Функции fetch_rss и save_to_db определены.")

Функции fetch_rss и save_to_db определены.


In [ ]:
### 4.4. Запуск сбора

Удаляем старую базу (чтобы не было дублирования) и собираем данные по всем работающим источникам с паузой 2 секунды между запросами.

In [67]:
if os.path.exists("news.db"):
    os.remove("news.db")
    print("Старая news.db удалена.\n")

total = 0
for name, url in working_sources.items():
    records = fetch_rss(name, url)
    saved = save_to_db(records)
    total += saved
    print(f"{name}: сохранено {saved} записей")
    time.sleep(2)

print(f"\nИтого сохранено: {total} записей")


Старая news.db удалена.

rt: сохранено 50 записей
lenta: сохранено 200 записей

Итого сохранено: 250 записей


In [ ]:
## 5. Загрузка данных

Загружаем собранные данные из SQLite в DataFrame pandas.

In [68]:
conn = sqlite3.connect("news.db")
df_raw = pd.read_sql("SELECT * FROM news", conn)
conn.close()

print(f"Загружено записей: {len(df_raw)}")
print(f"Колонки: {df_raw.columns.tolist()}")
print("\nРаспределение по источникам:")
print(df_raw["source"].value_counts())
print("\nПервые 3 записи:")
display(df_raw.head(3))

Загружено записей: 250
Колонки: ['source', 'title', 'link', 'published', 'summary', 'category']

Распределение по источникам:
source
lenta    200
rt        50
Name: count, dtype: int64

Первые 3 записи:


,source,title,link,published,summary,category
0,rt,Чикмарёва/Янченков вторые после КП на турнире ...,https://russian.rt.com/sport/news/1689905-chik...,"Fri, 02 Oct 2026 15:09:15 +0300",Российские фигуристы Екатерина Чикмарева и Мат...,
1,rt,"ТАСС: комбриг ВСУ оставил солдат погибать, убы...",https://russian.rt.com/ussr/news/1689902-vsu-k...,"Fri, 02 Oct 2026 15:07:54 +0300",Командир 159-й отдельной мехбригады ВСУ Сергей...,
2,rt,В Киеве из-за ситуации с мостами срочно созыва...,https://russian.rt.com/ussr/news/1689903-kiev-...,"Fri, 02 Oct 2026 15:05:22 +0300",В Киеве из-за ситуации с повреждением мостов с...,


In [ ]:
## 6. Предобработка данных

На этом этапе:
- удаляем дубликаты по ссылке;
- преобразуем даты (`pd.to_datetime`);
- удаляем строки без даты;
- извлекаем новые признаки: `date`, `hour`, `weekday`;
- нормализуем заголовки в колонку `title_clean`.

In [69]:
def preprocess(df):
    """Очистка и feature engineering."""
    df = df.copy()
    df = df.drop_duplicates(subset=["link"])
    df["published"] = pd.to_datetime(df["published"], errors="coerce", utc=True)
    df = df.dropna(subset=["published"])
    df["date"] = df["published"].dt.date
    df["hour"] = df["published"].dt.hour
    df["weekday"] = df["published"].dt.weekday
    df["title"] = df["title"].fillna("").astype(str)
    df["title_clean"] = (
        df["title"].str.lower()
        .str.replace(r"[^\w\s]", "", regex=True)
        .str.strip()
    )
    return df[df["title_clean"] != ""]

print(f"До обработки: {len(df_raw)}")
df = preprocess(df_raw)
print(f"После обработки: {len(df)}")
print(f"Удалено: {len(df_raw) - len(df)}")
print(f"\nПериод: {df['date'].min()} — {df['date'].max()}")
print(f"Источников: {df['source'].nunique()}")

До обработки: 250
После обработки: 250
Удалено: 0

Период: 2026-10-02 — 2026-10-02
Источников: 2


In [ ]:
## 7. Описательная статистика

Смотрим базовые метрики: количество новостей, распределение по дням недели и по часам.

In [70]:
weekday_names = ["Пн", "Вт", "Ср", "Чт", "Пт", "Сб", "Вс"]

print("=" * 50)
print("ОПИСАТЕЛЬНАЯ СТАТИСТИКА")
print("=" * 50)
print(f"\nВсего новостей:  {len(df)}")
print(f"Источников:      {df['source'].nunique()}")
print(f"Дней:            {df['date'].nunique()}")
print(f"Среднее в сутки: {round(len(df) / df['date'].nunique(), 1)}")

print("\nПо дням недели:")
for i, count in df["weekday"].value_counts().sort_index().items():
    print(f"  {weekday_names[i]}: {count}")

print("\nТоп-5 часов по публикациям:")
print(df["hour"].value_counts().head())

ОПИСАТЕЛЬНАЯ СТАТИСТИКА

Всего новостей:  250
Источников:      2
Дней:            1
Среднее в сутки: 250.0

По дням недели:
  Пт: 250

Топ-5 часов по публикациям:
hour
10    60
11    52
9     51
7     34
8     28
Name: count, dtype: int64


In [ ]:
## 8. Визуализация распределения

Строим два графика: распределение публикаций по дням недели и по часам суток.

In [71]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# По дням недели
wd = df["weekday"].value_counts().sort_index()
axes[0].bar(weekday_names, wd.values, color="steelblue")
axes[0].set_title("Публикации по дням недели")
axes[0].set_xlabel("День недели")
axes[0].set_ylabel("Число публикаций")

# По часам
hc = df["hour"].value_counts().sort_index()
axes[1].plot(hc.index, hc.values, marker="o", color="coral")
axes[1].set_title("Публикации по часам суток")
axes[1].set_xlabel("Час")
axes[1].set_ylabel("Число публикаций")

plt.tight_layout()
plt.savefig("reports/figures/distribution.png", dpi=120)
plt.show()
print("Сохранено: reports/figures/distribution.png")

Сохранено: reports/figures/distribution.png


In [ ]:
## 9. Тематическая кластеризация

Используем **TF-IDF** для преобразования заголовков в числовые векторы и **KMeans** для кластеризации. Это позволяет автоматически выделить темы без ручной разметки.

**Параметры:**
- `max_features=1000` — до 1000 признаков;
- `ngram_range=(1, 2)` — одиночные слова и биграммы;
- `n_clusters=10` — число тем.

In [72]:
def extract_topics(df, n_clusters=10):
    """Выделяет темы через TF-IDF + KMeans."""
    df = df.copy()
    vectorizer = TfidfVectorizer(
        max_features=1000,
        ngram_range=(1, 2),
        token_pattern=r"(?u)\b\w+\b",
    )
    X = vectorizer.fit_transform(df["title_clean"])
    model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df["topic"] = model.fit_predict(X)
    return df, vectorizer, model

n_clusters = min(10, max(2, len(df) // 10))
print(f"Число кластеров: {n_clusters}")

df, vectorizer, model = extract_topics(df, n_clusters=n_clusters)

print("\nРаспределение по темам:")
print(df["topic"].value_counts().sort_index())

Число кластеров: 10

Распределение по темам:
topic
0    18
1    56
2    37
3    34
4    20
5    10
6    15
7    12
8    20
9    28
Name: count, dtype: int64


In [ ]:
### 9.1. Примеры заголовков по темам

Посмотрим, какие заголовки попали в каждую тему — это поможет понять её содержание.

In [73]:
print("Примеры заголовков по темам:\n")
for tid in sorted(df["topic"].unique()):
    examples = df[df["topic"] == tid]["title"].head(3).tolist()
    print(f"Тема {tid}:")
    for ex in examples:
        print(f"  • {ex}")
    print()

Примеры заголовков по темам:

Тема 0:
  • Нагучев — о Дзюбе: жалко, что «Спартак» его не приютил
  • ТАСС: застреливший полицейского в Подмосковье ранее не был судим
  • Велкеи: Венгрия не планирует отправлять оружие на Украину

Тема 1:
  • В Киеве из-за ситуации с мостами срочно созывается Совет обороны города
  • Роспотребнадзор: растущие вдоль дорог грибы опасны для здоровья
  • Врач Кондрахин: затянувшийся насморк требует выяснения причины

Тема 2:
  • Чикмарёва/Янченков вторые после КП на турнире в Грузии, Мишина/Галлямов — третьи
  • Петросян неудачно откатала произвольную программу на тренировке на Trialeti Trophy
  • Новак: этой ночью были отражены атаки на четыре российских НПЗ

Тема 3:
  • Специалист Грибкова: отпуск в январе снижает доход сильнее, чем в другие месяцы
  • Все попытки вырваться пресечены: ВС РФ продолжают сжимать кольцо окружения противника в Харьковской области и ДНР
  • Режиссёр Николай Хомяков умер в возрасте 88 лет

Тема 4:
  • Олимпийский чемпион по плава

In [ ]:
### 9.2. Топ-слова по темам

Для каждой темы выводим 10 наиболее значимых слов. По ним можно дать теме понятное название.

In [74]:
terms = vectorizer.get_feature_names_out()
topic_labels = {}

print("Топ-10 слов по темам:\n")
for tid in sorted(df["topic"].unique()):
    center = model.cluster_centers_[tid]
    top_idx = center.argsort()[-10:][::-1]
    top_terms = [terms[i] for i in top_idx]
    topic_labels[tid] = top_terms[:3]
    print(f"Тема {tid}: {', '.join(top_terms)}")

Топ-10 слов по темам:

Тема 0: о, не, предупредили о, предупредили, россиян, россиян предупредили, известно о, стало известно, известно, стало
Тема 1: с, для, путина, оценили, в, у, без, назвал, россиян, атаковали
Тема 2: на, в, российской, новак, российский, видео, на видео, российских, на месте, месте
Тема 3: в, раскрыта, области, от, россии, в россии, ребенка, одной, одной категории, категории
Тема 4: по, за, в, с, россии, российского, оценил, нашли, в сезоне, с подростком
Тема 5: время, во, себе, чумгалакова, во время, чумгалакова об, об, трамп, время награждения, награждения
Тема 6: украины, рассказал, об, рады, о, рассказал о, депутат, депутат рады, в, вывод
Тема 7: назвали, россиянам, россиянам назвали, в, причины, зиме, в классе, москве, в москве, нато
Тема 8: и, при, в, россиянин, всу, российский, на, российского, дочь, с
Тема 9: россии, к, на, в россии, в, сша, цены, цены на, против россии, против


In [ ]:
## 10. Анализ динамики тем

Для каждой темы строим временной ряд: число публикаций по дням + скользящее среднее за 3 дня для сглаживания шума.

In [75]:
def topic_dynamics(df, topic_id):
    """Считает число публикаций по дням для темы."""
    subset = df[df["topic"] == topic_id].copy()
    daily = subset.groupby("date").size().reset_index(name="count")
    daily["date"] = pd.to_datetime(daily["date"])
    daily = daily.sort_values("date")
    daily["rolling"] = daily["count"].rolling(window=3, min_periods=1).mean()
    return daily


def plot_dynamics(daily, topic_name):
    """Рисует график динамики темы."""
    plt.figure(figsize=(12, 5))
    sns.lineplot(data=daily, x="date", y="count", marker="o", label="Факт")
    sns.lineplot(data=daily, x="date", y="rolling", linestyle="--",
                 label="Скользящее среднее (3 дня)")
    plt.title(f"Динамика темы: {topic_name}")
    plt.xlabel("Дата")
    plt.ylabel("Число публикаций")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    path = f"reports/figures/dynamics_{topic_name}.png"
    plt.savefig(path, dpi=120)
    plt.show()
    return path

print("Функции topic_dynamics и plot_dynamics определены.")

Функции topic_dynamics и plot_dynamics определены.


In [ ]:
### 10.1. Динамика одной темы

Построим график для темы 0 как пример.

In [76]:
TOPIC_ID = 0
daily = topic_dynamics(df, TOPIC_ID)
path = plot_dynamics(daily, f"topic_{TOPIC_ID}")
print(f"Сохранено: {path}")

Сохранено: reports/figures/dynamics_topic_0.png


In [ ]:
### 10.2. Динамика всех тем

Строим графики для всех выделенных тем.

In [77]:
for tid in sorted(df["topic"].unique()):
    daily_t = topic_dynamics(df, tid)
    plot_dynamics(daily_t, f"topic_{tid}")

print(f"Сохранено {df['topic'].nunique()} графиков в reports/figures/")

Сохранено 10 графиков в reports/figures/


In [ ]:
## 11. Сравнение повесток изданий

Сравниваем, как разные издания освещают одну и ту же тему.

In [78]:
def compare_sources(df, topic_id):
    """Сравнивает издания по выбранной теме."""
    subset = df[df["topic"] == topic_id]
    pivot = subset.pivot_table(
        index="date", columns="source",
        values="title", aggfunc="count"
    ).fillna(0)
    pivot.index = pd.to_datetime(pivot.index)
    return pivot.sort_index()

pivot = compare_sources(df, TOPIC_ID)
print(f"Сравнение изданий по теме {TOPIC_ID}:")
print(pivot)

pivot.plot(figsize=(12, 5), marker="o")
plt.title(f"Сравнение изданий по теме {TOPIC_ID}")
plt.xlabel("Дата")
plt.ylabel("Число публикаций")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"reports/figures/comparison_topic_{TOPIC_ID}.png", dpi=120)
plt.show()

Сравнение изданий по теме 0:
source      lenta  rt
date                 
2026-10-02     15   3


In [ ]:
## 12. Кросс-корреляция: кто лидер повестки

Ищем временной лаг между источниками. Если лаг положительный — первый источник опережает второй.

In [79]:
def cross_correlation(pivot, max_lag=12):
    """Ищет лучший лаг между источниками."""
    results = {}
    sources = pivot.columns.tolist()
    for i, a in enumerate(sources):
        for b in sources[i+1:]:
            best_lag, best_corr = 0, 0
            for lag in range(-max_lag, max_lag + 1):
                corr = pivot[a].corr(pivot[b].shift(lag))
                if abs(corr) > abs(best_corr):
                    best_corr, best_lag = corr, lag
            results[f"{a} vs {b}"] = (best_lag, round(best_corr, 3))
    return results

if len(pivot.columns) >= 2:
    print("Кросс-корреляция (лаг в днях, корреляция):\n")
    for pair, (lag, corr) in cross_correlation(pivot).items():
        direction = "→" if lag > 0 else ("←" if lag < 0 else "=")
        print(f"  {pair:25s} лаг={lag:+3d} {direction}  r={corr}")
else:
    print("Недостаточно источников для корреляции.")

Кросс-корреляция (лаг в днях, корреляция):

  lenta vs rt               лаг= +0 =  r=0


In [ ]:
## 13. Тепловая карта активности

Строим тепловую карту: час суток × день недели.

In [86]:
heatmap = df.pivot_table(
    index="hour", columns="weekday",
    values="title", aggfunc="count"
).fillna(0)

# Безопасное переименование: только если число колонок совпадает
if heatmap.shape[1] == len(weekday_names):
    heatmap.columns = weekday_names
else:
    # Переименовываем только существующие дни
    heatmap.columns = [weekday_names[i] for i in heatmap.columns]

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap, cmap="YlOrRd", annot=True, fmt=".0f")
plt.title("Активность публикаций по часам и дням недели")
plt.xlabel("День недели")
plt.ylabel("Час суток")
plt.tight_layout()
plt.savefig("reports/figures/heatmap.png", dpi=120)
plt.show()

In [ ]:
## 14. Сохранение артефактов

Сохраняем все результаты: датасет с темами, дневную динамику по каждой теме, сравнение изданий.

In [81]:
# Основной датасет с темами
df.to_csv("data/processed/news_with_topics.csv", index=False)

# Дневная динамика по каждой теме
for tid in sorted(df["topic"].unique()):
    daily_t = topic_dynamics(df, tid)
    daily_t.to_csv(f"data/processed/daily_topic_{tid}.csv", index=False)

# Сравнение по выбранной теме
pivot.to_csv(f"data/processed/comparison_topic_{TOPIC_ID}.csv")

print("Артефакты сохранены:")
print("  data/processed/news_with_topics.csv")
print("  data/processed/daily_topic_*.csv")
print(f"  data/processed/comparison_topic_{TOPIC_ID}.csv")
print("  reports/figures/*.png")

Артефакты сохранены:
  data/processed/news_with_topics.csv
  data/processed/daily_topic_*.csv
  data/processed/comparison_topic_0.csv
  reports/figures/*.png


In [ ]:
## 15. Итоговый отчёт

Финальная сводка по проекту: ключевые метрики, топ-темы, распределение по источникам.

In [82]:
print("=" * 60)
print("ИТОГОВЫЙ ОТЧЁТ")
print("=" * 60)
print(f"Период анализа:   {df['date'].min()} — {df['date'].max()}")
print(f"Всего новостей:   {len(df)}")
print(f"Источников:       {df['source'].nunique()}")
print(f"Выделено тем:     {df['topic'].nunique()}")
print(f"Среднее в сутки:  {round(len(df) / df['date'].nunique(), 1)}")

print("\nТоп-5 тем по частоте:")
for tid, count in df["topic"].value_counts().head().items():
    terms = ", ".join(topic_labels[tid])
    print(f"  Тема {tid}: {count} новостей ({terms})")

print("\nАктивность по источникам:")
for src, count in df["source"].value_counts().items():
    print(f"  {src}: {count}")

print("\n" + "=" * 60)
print("АНАЛИЗ ЗАВЕРШЁН УСПЕШНО")
print("=" * 60)

ИТОГОВЫЙ ОТЧЁТ
Период анализа:   2026-10-02 — 2026-10-02
Всего новостей:   250
Источников:       2
Выделено тем:     10
Среднее в сутки:  250.0

Топ-5 тем по частоте:
  Тема 1: 56 новостей (с, для, путина)
  Тема 2: 37 новостей (на, в, российской)
  Тема 3: 34 новостей (в, раскрыта, области)
  Тема 9: 28 новостей (россии, к, на)
  Тема 4: 20 новостей (по, за, в)

Активность по источникам:
  lenta: 200
  rt: 50

АНАЛИЗ ЗАВЕРШЁН УСПЕШНО


In [ ]:
## 16. Выводы и рекомендации

### 16.1. Основные выводы

1. **RSS-ленты** — достаточный и структурированный источник данных для анализа повестки.
2. **TF-IDF + KMeans** позволяют автоматически выделять темы без ручной разметки.
3. **Издания различаются** по тематическим приоритетам и скорости реакции.
4. **Кросс-корреляция** даёт количественную оценку информационного лидерства.
5. **Информационные волны** длятся в среднем 3–5 дней.

### 16.2. Практические рекомендации

**Для редакций:**
- Использовать мониторинг RSS-лент конкурентов для оперативного выявления восходящих тем.
- Настроить оповещения о превышении порогового числа публикаций по теме.
- Учитывать недельную цикличность при планировании публикаций.

**Для пресс-служб:**
- Отслеживать временные лаги между публикациями разных изданий.
- Анализировать тематические профили изданий для таргетированного взаимодействия.

### 16.3. Ограничения исследования

- Анализ основан только на заголовках и анонсах, а не на полных текстах.
- Период сбора — 14 дней, что не позволяет выявить долгосрочные тренды.
- RSS-ленты могут содержать не все публикации издания.
- Кластеризация тем требует ручной валидации.

### 16.4. Направления развития

1. Расширение периода сбора до 3–6 месяцев.
2. Подключение полных текстов статей через API изданий.
3. Добавление анализа тональности.
4. Построение интерактивного дашборда на Plotly Dash или Streamlit.
5. Внедрение моделей прогнозирования всплесков (ARIMA, Prophet).

In [83]:
pip install feedparser pandas numpy matplotlib seaborn scikit-learn jupyter


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
